### SoundSleep Muscle Contraction Scoring Script 

Script for scoring behavior responses during SoundSleep experiment using a combined human scorer/algorithm approach. 

#### Scoring Instructions

To score an epoch click the start and end locations of each individual contraction and the script will automatically calculate the number of contractions and the reaction times. The GUI will show the live binary scoring trace and contraction number. Press the next arrow to confirm the score. 

To rescore, any given epoch press "r". An epoch that has already been scored can not be rescored unless the "r" key is pressed. 

To mark an epoch as "0", click anywhere on the trace once. 

To reset the entire dataframe run the code in the "Reset" section. 

#### Scoring Algorithm 

Each epoch starts with a pre-calculated score from a scoring algorithm based on the variance of the pre-processed EMG signal. The variance is calculated in sliding windows of 50 samples.

The scoring algorithm takes a feature extracted from the pre-processed EMG signal. The current algorithm uses varaince, but one can also use root mean squared or waveform length as features. For a given epoch, there are two cases: no response and response. 

**No response case:** 
- checks if difference between baseline and max of the signal is less than a threshold 
- checks if the difference between the baseline and mean of the signal is less than a threshold 

**Response Case** 
- Loops over signal in a given epoch in a sliding window and checks if the mean variance in that window is greater than 1/4 of the max variance of the signal. 


If the scorer agrees with the scoring algorithm, the scorer can press the next key and the algorithm score will automatically be saved. If the scorer disagrees, they can rescore the epoch by either clicking anywhere on the epoch or pressing the "r" key. 



#### Flagging Functions 

Flags potential false alarms from scoring algorithm. Highlights periods that are too long in gray and periods that are too short in red. 

### Library importations 

In [1]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd

%matplotlib qt
import matplotlib
matplotlib.use('QtAgg') 
from matplotlib import pyplot as plt

### Global variables

In [20]:
# SUBJECT INFO - CHANGE EACH TIME 
subject = 'Cami'
path = '/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Data/SoundSleep_participants/Cami/'
filename = 'Cami_passive_wakr.vhdr'

In [21]:
baseline_window = 10 # number of samples for determining baseline
window = 50 # sliding window for features 
frq = 250
last_epoch_length = 5 # last epoch is five seconds 
num_epoch = 162 
epoch_scaling = 10000 # empirical scale for pre-processed EMG signal for plotting/algorithm 
step = 1

# Scoring algorithm parameters 
thr_max = 200 # max of feature threshold 
thr_mean = 10 # mean of feature threshold 
var_scale = 4 # scaling of variance for algorithm thresholding 

ch_used = 'Zygo' # change to Menton if there is an issue witj 

# globals for GUI 
overlay_line = None
signal_line = None
epoch_zygo_current = None
epoch_var_zygo_current = None
live_trace = np.zeros([])
scorer_labels = []

### Processing functions

In [22]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples
def get_features(epoch):
     # pad epoch to preserve sample number
     pad_left  = window // 2
     pad_right = window - 1 - pad_left  
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]

     var = np.var(epoch_sw, axis=-1) # calculate variance over window
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  
 
     return var, rms, wl

In [23]:
def contraction_from_feature(feature):
    # initialize binary trace for contraction detection 
    events = np.zeros(len(feature),dtype=bool)

    # baseline noise can either be at begining or end of epoch depending on where the trigger is  
    baseline_mean_start = np.mean(feature[:baseline_window])  
    baseline_mean_end = np.mean(feature[-baseline_window:-1])
    
    #----------NO RESPONSE CASE LOGIC--------------
    if ((abs(min(baseline_mean_start,baseline_mean_end) -
         np.max(feature))) < thr_max) or ((abs(min(baseline_mean_start,baseline_mean_end) -
         np.mean(feature))) < thr_mean):
        return events, 0 
    
    #----------RESPONSE CASE LOGIC--------------
    max_var = 0 
    for start in range(0, len(feature) - window + 1, 1):
        stop = start + window
        feature_win = feature[start:stop]
        curr_var = np.mean(feature_win)

        if (curr_var >  max_var): 
            max_var = curr_var 
    

    # find events that pass max feature 
    thr =   max_var/var_scale
    ind = np.where((feature> thr))
    events[ind] = True 

    
    return events,thr


In [24]:
# takes binary label array and returns start and end of signal 
def extract_periods(labels): 
    padded = np.pad(labels, (1, 1), constant_values=0)
    diff = np.diff(padded)

    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]

    lengths = ends - starts
    return lengths, starts, ends 

### Flagging Functions

In [25]:
# flags if the period between contractions is longer 
# set max period to be five seconds
def flag_period(labels,max_period=5):
    lengths,_,_ = extract_periods(labels) # gives length of each contraction 
    periods = lengths/frq 

    flagged_idx = np.where(periods > max_period)[0]
    if np.any(periods > max_period):
        return True,flagged_idx
    else:
        return False, None


In [26]:
# flags if the length of clusters is too short 
# set min period to be 100 samples 
def flag_clusters(labels,min_len=100):
    lengths, _, _ = extract_periods(labels)
    flagged_idx = np.where(lengths < min_len)[0]

    if np.any(lengths < min_len):
        return True, flagged_idx
    else:
        return False, None

### Plotting functions 

In [27]:
# loads current epoch based on index 
def load_current_epoch():
    global epoch_zygo_current, epoch_var_zygo_current, raw,last_epoch_length,current_index,df_triggers 
    
    if current_index == num_epoch-1:
        raw_zygo = raw.get_data(picks=[ch_used],start=int(df_triggers['Stim_time_sample'][current_index]-1),stop=int(df_triggers['Stim_time_sample'][current_index]+last_epoch_length))
    else:
        raw_zygo = raw.get_data(picks=[ch_used],start=int(df_triggers['Stim_time_sample'][current_index]-1),stop=int(df_triggers['Stim_time_sample'][current_index+1]-1)) # logic for last epoch 
   
    raw_zygo = raw_zygo[0] * epoch_scaling
    epoch_var_zygo_current, _, _ = get_features(raw_zygo) # returns variance of raw signal 
    epoch_zygo_current = raw_zygo # set global variable 
    

In [28]:
# determines score for current epoch 
def resolve_epoch_score(feature_zygo, current_index):
    scorer_val = df_triggers.at[current_index, 'Scorer']

    if scorer_val == 'human' or scorer_val == 'algorithm': # loads score if epoch already visited 
        zygo_out = df_triggers.at[current_index, 'labels']
        scorer_str = scorer_val
    elif len(scorer_labels) > 0: # loasd score if human rescoring 
        zygo_out = live_trace
        scorer_str = 'human'
    else: # loads algorithm score if not yet visited 
        zygo_out, _ = contraction_from_feature(feature_zygo) # using variance as feature 
        zygo_out = zygo_out.astype(int)
        scorer_str = 'algorithm'

    lengths, starts, ends = extract_periods(zygo_out)
    return zygo_out, scorer_str, lengths, starts, ends

In [29]:
# builds binary trace for plotting based on clicked markers from key press 
def build_trace(epoch_length, markers):
    trace = np.zeros(epoch_length, dtype=int)
    complete_pairs = len(markers) // 2
    for i in range(complete_pairs):
        s, e = markers[2 * i], markers[2 * i + 1]
        s, e = int(min(s, e)), int(max(s, e))
        s, e = max(s, 0), min(e, epoch_length - 1)
        trace[s:e + 1] = 1
    return trace

In [30]:
# GUI key press function for scoring contraction 
def on_key(event):
    global current_index, fig, subject_epoch,scorer_labels, df_triggers, scorer_line, pending_marker,epoch_zygo_current

    if event.key == 'right': # calculates epoch score and saves current state of epoch to df 
        zygo_out, scorer_str, lengths, starts, ends = resolve_epoch_score(epoch_var_zygo_current, current_index)

        df_triggers.at[current_index, 'labels'] = zygo_out
        df_triggers.at[current_index, 'Scorer'] = scorer_str
        df_triggers.at[current_index, 'Contraction_number'] = len(lengths)
        if len(starts) == 0: # if 0 contractions, saves RT, start, and end as nan 
            df_triggers.loc[current_index, ['RT_sec', 'start', 'end']] = np.nan
        else: 
            df_triggers.at[current_index, 'RT_sec'] = starts[0] / frq
            df_triggers.at[current_index, 'start'] = starts[0] # or not the indexing to add all?
            df_triggers.at[current_index, 'end'] = ends[-1]

        # resets scorer labels and increase index 
        scorer_labels = []
        current_index = (current_index + 1) % len(subject_epoch)
        plt.close()
        load_current_epoch()
        fig, _ = plot_binary_trace(epoch_zygo_current, epoch_var_zygo_current) # plots current epoch 


    elif event.key == 'left': # loads previous epoch  
        scorer_labels = [] 
        current_index = (current_index - 1) % len(df_triggers) # Loop to the end   
        plt.close()
        load_current_epoch()
        fig,zygo_out = plot_binary_trace(epoch_zygo_current, epoch_var_zygo_current)
        lengths, starts, ends = extract_periods(zygo_out )

    elif event.key == 'r':  # resets epoch scores 
        df_triggers.loc[current_index, ['start', 'end', 'Contraction_number', 'RT_sec']] = np.nan
        df_triggers.loc[current_index, 'labels'] = None
        df_triggers.loc[current_index, 'Scorer'] = 'reset'   # tri-state sentinel, not None
        scorer_labels = []
        live_trace = np.zeros(len(epoch_zygo_current), dtype=int)

        plt.close()
        load_current_epoch()
        fig, _ = plot_binary_trace(epoch_zygo_current, epoch_var_zygo_current)


    load_current_epoch()   

In [31]:
# mouse click logic for re-scoring epoch  
def on_click(event):
    global current_index, df_triggers, live_trace, overlay_line, fig,epoch_var_zygo_current,epoch_zygo_current 

    if not event.inaxes:
        return

    epoch_length = np.shape(epoch_zygo_current)[0]

    # calculating time based on which epoch 
    if current_index == num_epoch-1:
        time = np.linspace(0, df_triggers['Stim_time_sec'][current_index] +5, epoch_length)

    else:
        time = np.linspace(0, df_triggers['Stim_time_sec'][current_index+1] - df_triggers['Stim_time_sec'][current_index]-2/frq, epoch_length)

    click_sample = int(round(event.xdata * frq))  # xdata is in seconds
    scorer_labels.append(click_sample) # add sample to the scorer data 

    live_trace = build_trace(epoch_length, scorer_labels) # creates binary trace from where scorer clicked 
    
    # Live overlay of binary trace as scoring is happening 
    if len(scorer_labels) % 2 == 1:
        fig.suptitle(f"Epoch: {current_index} | DETECTED: {int(len(scorer_labels)/2)} | SCORING IN PROGRESS", color="orange")
    else:
        fig.suptitle(f"Epoch: {current_index} | DETECTED: {int(len(scorer_labels)/2)} | SCORED BY HUMAN", color="green")

    # plots overlay line 
    overlay_line.set_data(time, live_trace)
    overlay_line.set_color("green")
    fig.canvas.draw_idle()


In [32]:
# plots binary trace calculated from signal 
def plot_binary_trace(zygo,feature_zygo):
    global current_index, live_trace,fig, df_triggers,  overlay_line, signal_line 

    fig, ax = plt.subplots(1, 1, figsize=(8, 4), sharex=True)
    ax_zygo = ax.twinx()
    zygomatic_color = "#6F9359"

    # calculate time based on start vs end of epoch 
    if current_index == num_epoch-1:
        time = np.linspace(0, df_triggers['Stim_time_sec'][current_index] +5, len(zygo))
    else:
        time = np.linspace(0, df_triggers['Stim_time_sec'][current_index+1] - df_triggers['Stim_time_sec'][current_index]-2/frq, len(zygo))

    scorer_val = df_triggers.at[current_index, 'Scorer'] # was epoch scored by human or algorithm 

    if pd.isna(scorer_val) or scorer_val is None:
        # if epoch never visited: show algorithm's first guess
        zygo_out, _ = contraction_from_feature(feature_zygo) # using variance as feature 
        zygo_out = zygo_out.astype(int)

        lengths, starts, ends = extract_periods(zygo_out)
        cluster_flg, flagged_idx_c = flag_clusters(zygo_out, 30)
        period_flg, flagged_idx_p = flag_period(zygo_out, .5)

        # plot flags for algorithm 
        if (period_flg or cluster_flg):
            if (period_flg):
                title = fig.suptitle(f"Epoch: {current_index} | DETECTED: {len(lengths)} | FlAGGED LABELS", color="red")
                for i in flagged_idx_p:
                    ax.axvspan(starts[i] / frq, ends[i] / frq, color="grey", alpha=0.3, label="long cluster flag")
            if (cluster_flg):
                title = fig.suptitle(f"Epoch: {current_index} | DETECTED: {len(lengths)} | FlAGGED LABELS", color="red")
                for i in flagged_idx_c:
                    ax.axvspan(starts[i] / frq, ends[i] / frq, color="red", alpha=0.3, label="short cluster flag")
        else:
            title = fig.suptitle(f"Epoch: {current_index} | DETECTED: {len(lengths)} | OK LABELS", color="black")

        line_color = "red"
        label_text = "computed labels"

    # scorer reset epoch scores 
    elif scorer_val == 'reset':
        if len(scorer_labels) > 0:
            zygo_out = live_trace
            lengths, starts, ends = extract_periods(live_trace)
            title = fig.suptitle(
                f"Epoch: {current_index} | DETECTED: {int(len(scorer_labels)/2)} | SCORED BY HUMAN",
                color="green"
            )
        else:
            zygo_out = np.zeros(len(zygo), dtype=int)
            title = fig.suptitle(f"Epoch: {current_index} | RESET — click to re-score", color="orange")
        line_color = "green"
        label_text = "scorer labels"

    else: # loads existing scores/algorithm trace in epoch
        zygo_out = df_triggers.at[current_index, 'labels']
        scorer = df_triggers.at[current_index, 'Scorer']
        contraction_num = df_triggers.at[current_index, 'Contraction_number']

        line_color = "green" if (scorer == 'human') else "red"
        label_text = "scorer labels" if (scorer == 'human') else "computed labels"
        text = "SCORED human" if (scorer == 'human') else "SCORED algo"

        title = fig.suptitle(f"Epoch: {current_index} | DETECTED: {contraction_num} | {text}",color='green')
    
    signal_line, = ax.plot(time, zygo, lw=1.5, color=zygomatic_color)
    ax.set_ylim(-200, 200)
    ax.set_ylabel("Zygo",size=12)


    fig.canvas.mpl_connect('key_press_event', on_key)
    fig.canvas.mpl_connect('button_press_event', on_click)

    overlay_line, = ax_zygo.plot(time, zygo_out, color=line_color, label=label_text, alpha=0.7)
    ax_zygo.axis("on")
    plt.legend() 

    return fig, zygo_out

### Pre-processing 

In [33]:
raw = mne.io.read_raw_brainvision(path+filename, preload=True)
emg_ch= ['Zygo', 'Menton']


if subject=='Cami':  # For Cami, EOG was recorded on IO channel
    mne.rename_channels(info=raw.info,mapping={'66':'ECG' ,'67':'Menton','68':'Zygo'})
    raw.set_channel_types(mapping={'ECG':'ecg','Zygo':'emg','Menton':'emg','IO':'eog'})
    mne.add_reference_channels(raw, ref_channels=['C3'], copy=False)
else:
    mne.rename_channels(info=raw.info,mapping={'66':'ECG' ,'67':'Menton','68':'Zygo','69':'EOG'})
    raw.set_channel_types(mapping={'ECG':'ecg','Zygo':'emg','Menton':'emg','IO':'eog','EOG':'eog'})  
    mne.add_reference_channels(raw, ref_channels=['Cz'], copy=False)  
if subject=='S02':
    idx=raw.ch_names.index('EOG')
    raw._data[idx]*=-1
    
#raw = mne.set_bipolar_reference(raw, anode='IO', cathode='AF7',drop_refs=False) #
raw.resample(sfreq=250)

#raw.set_eeg_reference(['TP10'], projection=False) 

emg_filter_params = {'lpass': 100,'hpass': 10,'notches': [50]}
eeg_eog_filter_params = {'lpass': 15,'hpass': 0.3,'notches': [50]}
ecg_filter_params = {'lpass': 70,'hpass': 0.3,'notches': [50]}

raw.filter(l_freq=emg_filter_params['hpass'],h_freq=emg_filter_params['lpass'],picks=emg_ch)
raw.notch_filter(emg_filter_params['notches'],picks=emg_ch)
#raw.filter(l_freq=eeg_eog_filter_params['hpass'],h_freq=eeg_eog_filter_params['lpass'],picks=mne.pick_types(raw.info, eeg=True, eog=True))
#raw.filter(l_freq=ecg_filter_params['hpass'],h_freq=ecg_filter_params['lpass'],picks=mne.pick_types(raw.info, ecg=True))

events_all= mne.events_from_annotations(raw)[0]
events= mne.pick_events(events_all,include=list(range(1,40))+[99]) # Pick events of interest (where a stimulus was presented)

df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
df_triggers.drop(columns=['dunno'],inplace=True)
df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


# Create trial type column 
conditions = [
    df_triggers['Trigger'].between(1,10),
    df_triggers['Trigger'].between(11,20),
    df_triggers['Trigger'].between(21,30),
    df_triggers['Trigger'].between(31,40),
    df_triggers['Trigger'] == 99 
]

choices = ['SNR1','SNR2','SNR3','blank', 'Tone'] # trial types 
df_triggers['trial_type'] = np.select(conditions, choices, default='unknown')

epochs = mne.Epochs(raw, events, tmin=-0, tmax=7,baseline=None, detrend=0,
                reject=None, preload=True, on_missing='warn')

subject_epoch = epochs 

Extracting parameters from /Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Data/SoundSleep_participants/Cami/Cami_passive_wakr.vhdr...
Setting channel info structure...
Reading 0 ... 3443099  =      0.000 ...  1377.240 secs...


/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_87971/3229227733.py:1: RuntimeWarning: No coordinate information found for channels ['IO', '65', '66', '67', '68', '69', '70', '71', '72']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(path+filename, preload=True)
/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_87971/3229227733.py:1: RuntimeWarning: Not setting positions of 9 misc channels found in montage:
['IO', '65', '66', '67', '68', '69', '70', '71', '72']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(path+filename, preload=True)
/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_87971/3229227733.py:7: RuntimeWarning: The unit for channel(s) ECG, IO, Menton, Zygo has changed from NA to V.
  raw.set_channel_types(

Filtering a subset of channels. The highpass and lowpass values in the measurement info will not be updated.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 1e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 100.00 Hz
- Upper transition bandwidth: 25.00 Hz (-6 dB cutoff frequency: 112.50 Hz)
- Filter length: 331 samples (1.324 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower

In [34]:
# initializing dataframe 

df_triggers['RT_sec'] = np.nan
df_triggers['Contraction_number'] = np.nan
df_triggers['start'] = np.nan 
df_triggers['end'] = np.nan 

df_triggers['labels'] = None
df_triggers['Scorer'] = None
df_triggers = df_triggers.rename(columns={
    "Time(Sample)": "Stim_time_sample",
    "Time(s)": "Stim_time_sec"})

### Scoring GUI

In [35]:
# extract epoch  
current_index = 0

# logic for last epoch take the last 5 seconds of signal 
if current_index == num_epoch-1:
    epoch_zygo_current = raw.get_data(picks=[ch_used],start=int(df_triggers['Stim_time_sample'][current_index]-1),stop=int(df_triggers['Stim_time_sample'][current_index]+last_epoch_length))

else:
    epoch_zygo_current = raw.get_data(picks=[ch_used],start=int(df_triggers['Stim_time_sample'][current_index]-1),stop=int(df_triggers['Stim_time_sample'][current_index+1]-1))
    
    
epoch_zygo_current = epoch_zygo_current[0]*epoch_scaling # multiply by 10000 for scalign 


# get features for epoch 
epoch_var_zygo_current, epoch_rms_zygo, epoch_wl_zygo  = get_features(epoch_zygo_current)


fig,_= plot_binary_trace(epoch_zygo_current,epoch_var_zygo_current)

In [36]:
# rename and reorder columns to match CSV 
df_triggers = df_triggers.rename(columns={
    "Time(Sample)": "Stim_time_sample",
    "Time(s)": "Stim_time_sec", 
    "start": "Response_start_sample",
    "end": "Response_end_sample"
})

df_triggers = df_triggers[["Stim_time_sample", "Trigger", "Stim_time_sec",
                           "Contraction_number","Response_start_sample","Response_end_sample","RT_sec","trial_type","Scorer","labels"]]

In [37]:
# saving df 
df_triggers.to_csv(f'{path}subject_{subject}_wake_metadata.csv'.format(subject),index=False)

### Reseting data frame 

In [ ]:
# reset df 
df_triggers[['labels']] = None
df_triggers[["RT_sec", "start","end","Scorer"]] = np.nan